# Name That LLM: Fine-tune DeBERTa-v3

Fine-tunes `microsoft/deberta-v3-base` as an 18-way classifier and reports **balanced accuracy** on the same stratified holdout (seed 42, 20%) as the traditional-models notebook, so the numbers are directly comparable.

**Design choices**
* **Newline token.** DeBERTa's SentencePiece tokenizer collapses line breaks, but formatting (lists, paragraphs, markdown) is a strong style signal. Newlines are replaced with a dedicated `[NL]` token.
* **Head + tail truncation.** Long passages keep their beginning *and* end (openings and sign-offs are very telling) instead of cutting off the end.
* **Plain PyTorch loop** with mixed precision, dynamic padding, linear warm-up/decay, best-epoch checkpointing and an optional time budget.
* Saves holdout and test **probabilities** (`.npy`) so the model can be blended with other models later.

**Running on Kaggle:** code-competition notebooks usually have no internet. Add the model as an input (Add Input → Models → DeBERTa-v3) and set `MODEL_NAME` to its path, e.g. `/kaggle/input/deberta-v3/transformers/base/1`. Use a GPU (T4 x2 / P100 / better).

Smoke test first: `MAX_TRAIN = 2000`, `EPOCHS = 1`.

In [ ]:
import os, glob, time, math, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
SEED = 42

# ---------------- CONFIG ----------------
MODEL_NAME  = os.environ.get("NTL_MODEL") or "microsoft/deberta-v3-base"   # or a local path
MAX_TRAIN   = None      # e.g. 2000 for a smoke test
MAX_LEN     = 512       # tokens per passage incl. [CLS]/[SEP]
HEAD_FRAC   = 0.75      # share of the token budget taken from the start when truncating
NEWLINE_TOK = True      # replace "\n" with a dedicated [NL] token
EPOCHS      = 4
BATCH       = 8         # per-step batch; lower to 4 if you hit out-of-memory
ACCUM       = 2         # gradient accumulation -> effective batch = BATCH * ACCUM
LR          = 2e-5
WARMUP      = 0.1
WEIGHT_DECAY = 0.01
HOLDOUT     = 0.2
TIME_BUDGET_MIN = None  # stop training gracefully after this many minutes (None = no limit)
TEXT_COL = LABEL_COL = ID_COL = None   # None = auto-detect
CANDIDATE_DIRS = [os.environ.get("NTL_DATA_DIR"), "/kaggle/input/competitions/name-that-llm",
                  "/kaggle/input/name-that-llm", "data"]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (training will be very slow)")

## 1. Load data (same logic as the traditional-models notebook)

In [ ]:
def find_file(root, stem):
    for ext in ("csv", "parquet"):
        hits = glob.glob(os.path.join(root, "**", f"{stem}*.{ext}"), recursive=True)
        if stem != "sample_submission":
            hits = [h for h in hits if "sample" not in os.path.basename(h).lower()]
        if hits:
            return sorted(hits)[0]
    return None

def find_root():
    for d in CANDIDATE_DIRS:
        if d and os.path.isdir(d) and find_file(d, "train"):
            return d
    import kagglehub
    return kagglehub.competition_download("name-that-llm")

read_any = lambda p: pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p)

ROOT = find_root()
train, test = read_any(find_file(ROOT, "train")), read_any(find_file(ROOT, "test"))
sub_path = find_file(ROOT, "sample_submission")

def detect_cols(train, test):
    label = LABEL_COL or next((c for c in ["label","labels","model","model_name","target","llm","generator","source","y"]
                               if c in train.columns), None)
    if label is None:
        extra = [c for c in train.columns if c not in test.columns]
        label = extra[0] if extra else None
    obj = [c for c in train.columns if c != label and train[c].dtype == object]
    text = TEXT_COL or next((c for c in ["text","passage","response","answer","content","output","completion"]
                             if c in train.columns), None) or max(obj, key=lambda c: train[c].astype(str).str.len().mean())
    idc = ID_COL or next((c for c in ["id","ID","row_id","idx","index"] if c in test.columns), None)
    return text, label, idc

TEXT_COL, LABEL_COL, ID_COL = detect_cols(train, test)
print(f"TEXT_COL={TEXT_COL!r} LABEL_COL={LABEL_COL!r} ID_COL={ID_COL!r} | train {train.shape} test {test.shape}")

train = train.dropna(subset=[LABEL_COL]).reset_index(drop=True)
if MAX_TRAIN and len(train) > MAX_TRAIN:
    train = train.groupby(LABEL_COL, group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, MAX_TRAIN * len(g) // len(train))), random_state=SEED)).reset_index(drop=True)

le = LabelEncoder()
y = le.fit_transform(train[LABEL_COL])
CLASSES, n_cls, n_train = le.classes_, len(le.classes_), len(train)
train_text = train[TEXT_COL].fillna("").astype(str).tolist()
test_text = test[TEXT_COL].fillna("").astype(str).tolist()
print(n_cls, "classes,", n_train, "training rows")

# identical split to the traditional-models notebook (same seed / size / stratification)
idx_tr, idx_va = train_test_split(np.arange(n_train), test_size=HOLDOUT, stratify=y, random_state=SEED)
print("train", len(idx_tr), "| holdout", len(idx_va))

## 2. Tokenizer, model and tokenization

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if NEWLINE_TOK:
    tok.add_tokens(["[NL]"])
NL = " [NL] "

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_cls)
if NEWLINE_TOK:
    model.resize_token_embeddings(len(tok))

CLS = tok.cls_token_id if tok.cls_token_id is not None else tok.bos_token_id
SEP = tok.sep_token_id if tok.sep_token_id is not None else tok.eos_token_id
PAD = tok.pad_token_id if tok.pad_token_id is not None else 0

def encode(texts, bs=2000):
    """Tokenise without truncation, then keep head+tail tokens and add [CLS]/[SEP]."""
    budget = MAX_LEN - 2
    head = int(budget * HEAD_FRAC); tail = budget - head
    out = []
    for i in range(0, len(texts), bs):
        chunk = texts[i:i + bs]
        if NEWLINE_TOK:
            chunk = [t.replace("\r\n", "\n").replace("\n", NL) for t in chunk]
        for ids in tok(chunk, add_special_tokens=False, truncation=False)["input_ids"]:
            if len(ids) > budget:
                ids = ids[:head] + ids[-tail:]
            out.append([CLS] + ids + [SEP])
    return out

t0 = time.time()
enc_train = encode(train_text)
enc_test = encode(test_text)
lens = np.array([len(x) for x in enc_train])
print(f"tokenised in {time.time()-t0:.0f}s | token length: median {np.median(lens):.0f}, "
      f"p90 {np.percentile(lens, 90):.0f}, max {lens.max()} | truncated: {(lens >= MAX_LEN).mean():.1%}")
print("sample:", tok.decode(enc_train[0][:40]))

In [ ]:
class DS(Dataset):
    def __init__(self, enc, labels=None): self.enc, self.labels = enc, labels
    def __len__(self): return len(self.enc)
    def __getitem__(self, i):
        return self.enc[i], (-1 if self.labels is None else int(self.labels[i]))

def collate(batch):
    L = max(len(x) for x, _ in batch)
    ids = torch.full((len(batch), L), PAD, dtype=torch.long)
    mask = torch.zeros((len(batch), L), dtype=torch.long)
    for i, (x, _) in enumerate(batch):
        ids[i, :len(x)] = torch.tensor(x); mask[i, :len(x)] = 1
    return ids, mask, torch.tensor([l for _, l in batch])

tr_ds = DS([enc_train[i] for i in idx_tr], y[idx_tr])
train_loader = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2, drop_last=False)

@torch.no_grad()
def predict(enc, bs=32):
    """Softmax probabilities in the original order (length-sorted batches for speed)."""
    model.eval()
    order = np.argsort([len(x) for x in enc])
    probs = np.zeros((len(enc), n_cls), dtype=np.float32)
    for s in range(0, len(order), bs):
        sel = order[s:s + bs]
        ids, mask, _ = collate([(enc[i], -1) for i in sel])
        with torch.autocast(device, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
        probs[sel] = torch.softmax(logits.float(), -1).cpu().numpy()
    return probs

## 3. Train

In [ ]:
USE_AMP = device == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and AMP_DTYPE == torch.float16)

model.to(device)
no_decay = ("bias", "LayerNorm.weight")
opt = torch.optim.AdamW([
    {"params": [p for n, p in model.named_parameters() if not any(k in n for k in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(k in n for k in no_decay)], "weight_decay": 0.0},
], lr=LR)

steps_per_epoch = math.ceil(len(train_loader) / ACCUM)
total_steps = steps_per_epoch * EPOCHS
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total_steps), total_steps)
print(f"AMP={USE_AMP} ({AMP_DTYPE}) | {steps_per_epoch} optimizer steps/epoch, {total_steps} total")

CKPT = "best_model.pt"
history, best, t_start, stop = [], -1.0, time.time(), False
y_va = y[idx_va]
enc_va = [enc_train[i] for i in idx_va]

for epoch in range(1, EPOCHS + 1):
    model.train(); run_loss, n_seen = 0.0, 0
    opt.zero_grad(set_to_none=True)
    for step, (ids, mask, lab) in enumerate(train_loader, 1):
        with torch.autocast(device, dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=lab.to(device))
        loss = out.loss / ACCUM
        scaler.scale(loss).backward()
        run_loss += out.loss.item() * len(lab); n_seen += len(lab)
        if step % ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
        if epoch == 1 and step == 40:
            per = (time.time() - t_start) / 40
            print(f"  speed: {per:.2f}s/step -> ~{per * len(train_loader) / 60:.1f} min/epoch")
        if TIME_BUDGET_MIN and (time.time() - t_start) / 60 > TIME_BUDGET_MIN:
            stop = True; break

    P_va = predict(enc_va)
    pred = P_va.argmax(1)
    row = dict(epoch=epoch, train_loss=run_loss / max(n_seen, 1),
               bal_acc=balanced_accuracy_score(y_va, pred), acc=accuracy_score(y_va, pred),
               macro_f1=f1_score(y_va, pred, average="macro"), minutes=(time.time() - t_start) / 60)
    history.append(row)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
    if row["bal_acc"] > best:
        best = row["bal_acc"]; torch.save(model.state_dict(), CKPT); best_P_va = P_va
        print("  saved best checkpoint")
    if stop:
        print("time budget reached, stopping"); break

hist = pd.DataFrame(history)
print("\nBest holdout balanced accuracy:", round(best, 4))
hist.round(4)

## 4. Evaluate, save probabilities, write submission

In [ ]:
model.load_state_dict(torch.load(CKPT, map_location=device))
P_va = best_P_va
pred = P_va.argmax(1)
print("holdout balanced acc:", round(balanced_accuracy_score(y_va, pred), 4),
      "| accuracy:", round(accuracy_score(y_va, pred), 4),
      "| macro F1:", round(f1_score(y_va, pred, average="macro"), 4))

recall = pd.Series(confusion_matrix(y_va, pred, normalize="true").diagonal(), index=CLASSES).sort_values()
print("\nPer-class recall (weakest first):"); print(recall.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(y_va, pred, display_labels=CLASSES, normalize="true",
                                        xticks_rotation=90, values_format=".2f", ax=ax, colorbar=False)
ax.set_title("DeBERTa holdout confusion matrix (row-normalised)"); plt.tight_layout(); plt.show()

In [ ]:
# Probabilities for later blending (holdout rows are in idx_va order)
P_test = predict(enc_test)
np.save("deberta_holdout_probs.npy", P_va); np.save("deberta_holdout_idx.npy", idx_va)
np.save("deberta_test_probs.npy", P_test)
pd.Series(list(CLASSES)).to_csv("deberta_classes.csv", index=False, header=False)

test_pred = CLASSES[P_test.argmax(1)]
ids = test[ID_COL] if ID_COL else pd.Series(np.arange(len(test)), name="id")
if sub_path:
    cols = list(read_any(sub_path).columns)
    if set(map(str, CLASSES)) <= set(cols):
        sub = pd.DataFrame(P_test, columns=list(map(str, CLASSES))); sub.insert(0, cols[0], ids.values)
    else:
        sub = pd.DataFrame({cols[0]: ids.values, cols[1]: test_pred})
else:
    sub = pd.DataFrame({ID_COL or "id": ids.values, LABEL_COL: test_pred})
sub.to_parquet("submission.parquet", index=False)
print(sub.shape); sub.head()